# 🟢 Atividade 01 — Tokens: Como a IA lê texto

**Conceito:** A IA não lê letras nem palavras inteiras. Ela fatia o texto em *tokens* — pedaços de palavras que podem ser sílabas, prefixos ou palavras inteiras.

**Objetivo desta atividade:**
- Entender o que são tokens e como o texto é dividido
- Comparar o uso de tokens entre idiomas
- Usar a API do Claude para contar tokens oficialmente
- Completar o desafio de programação no final

---
> ⚠️ **Antes de começar:** substitua `SUA_CHAVE_AQUI` na célula de setup pela sua chave da API Anthropic.
> Acesse https://console.anthropic.com para criar uma chave gratuita.

## ⚙️ Setup — Rode esta célula primeiro!

In [ ]:
# Instala as bibliotecas necessárias
%pip install anthropic tiktoken -q

import anthropic
import tiktoken
import os
from typing import Dict, Any

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO - Edite apenas esta seção
# ═══════════════════════════════════════════════════════════════════════════
os.environ["ANTHROPIC_API_KEY"] = "SUA_CHAVE_AQUI"  # ← Sua chave aqui
MODEL_NAME = "claude-sonnet-4-6"  # Modelo a usar (claude-sonnet-4-6, claude-opus-4, etc.)
# ═══════════════════════════════════════════════════════════════════════════

client = anthropic.Anthropic()

# Teste de conexão com tratamento de erro
try:
    teste = client.messages.create(
        model=MODEL_NAME,
        max_tokens=20,
        messages=[{"role": "user", "content": "Diga apenas: funcionou!"}]
    )
    print(" Status:", teste.content[0].text)
except anthropic.AuthenticationError:
    print(" Erro: Chave de API inválida. Verifique sua ANTHROPIC_API_KEY.")
except anthropic.RateLimitError:
    print(" Erro: Rate limit atingido. Aguarde um momento e tente novamente.")
except Exception as e:
    print(f" Erro inesperado: {e}")

---
## 📖 Parte 1 — Visualizando tokens com tiktoken

O `tiktoken` é a biblioteca da OpenAI para tokenização — usa um encoding parecido com o do Claude.
Vamos ver como diferentes frases são divididas em pedaços.

> ⚠️ **Nota importante:** O tiktoken usa o tokenizador da OpenAI (cl100k_base), não o tokenizador oficial
> do Claude. Os resultados são aproximados! Na Parte 2 usaremos o contador oficial da Anthropic.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

frases = [
    "gato",
    "cachorro",
    "inteligência",
    "Olá, tudo bem?",
    "Hello, how are you?",
    "A fotossíntese é o processo pelo qual as plantas produzem energia.",
    "Photosynthesis is the process by which plants produce energy.",
    "supercalifragilisticexpialidocious",
    "IA",
    "inteligência artificial",
    "artificial intelligence",
]

print(f"{'Frase':<58} {'Tokens':>6}  Pedaços")
print("─" * 100)
for frase in frases:
    tokens = enc.encode(frase)
    pedacos = [enc.decode([t]) for t in tokens]
    print(f"{frase:<58} {len(tokens):>6}  {pedacos}")

In [ ]:
# ─── Comparação: tiktoken vs contador oficial do Claude ───
# Vamos ver a diferença entre o tokenizador da OpenAI e o oficial da Anthropic

print("🔍 COMPARANDO TOKENIZADORES: tiktoken (OpenAI) vs Claude (oficial)")
print("=" * 75)
print(f"{'Frase':<40} {'tiktoken':>10} {'Claude':>10} {'Diferença':>12}")
print("─" * 75)

frases_comparacao = [
    "inteligência artificial",
    "Hello, world!",
    "fotossíntese",
    "supercalifragilisticexpialidocious",
]

try:
    for frase in frases_comparacao:
        # tiktoken (OpenAI)
        tokens_tiktoken = len(enc.encode(frase))
        
        # Claude oficial
        contagem = client.messages.count_tokens(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": frase}]
        )
        tokens_claude = contagem.input_tokens
        
        diff = tokens_claude - tokens_tiktoken
        diff_str = f"+{diff}" if diff > 0 else str(diff)
        print(f"{frase:<40} {tokens_tiktoken:>10} {tokens_claude:>10} {diff_str:>12}")
except Exception as e:
    print(f"❌ Erro: {e}")

print("\n💡 Conclusão: use sempre o contador oficial para cálculos de custo!")

### 🤔 Observe e reflita:
- Qual palavra usou mais tokens? Faz sentido?
- Compare `"IA"` vs `"inteligência artificial"` — o que você nota?
- Por que `"Photosynthesis"` e `"fotossíntese"` têm quantidades de tokens diferentes?

---
## 🔬 Parte 2 — Contador oficial do Claude

A API da Anthropic tem um endpoint específico para contar tokens **antes** de fazer a chamada. Isso é útil para controlar custos!

In [ ]:
textos = [
    "Olá, mundo!",
    "Hello, world!",
    "A inteligência artificial está mudando tudo.",
    "Artificial intelligence is changing everything.",
    "O rato roeu a roupa do rei de Roma.",  # trava-língua!
]

print(f"{'Texto':<50} {'Tokens Claude':>14}")
print("─" * 66)

try:
    for texto in textos:
        contagem = client.messages.count_tokens(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": texto}]
        )
        print(f"{texto:<50} {contagem.input_tokens:>14}")
except Exception as e:
    print(f"❌ Erro ao contar tokens: {e}")

print("\n💡 Dica: o Claude cobra por token de entrada E saída!")

---
## 💡 Parte 3 — Por que tokens importam para o custo?

Vamos simular quanto custaria processar diferentes textos.

In [ ]:
# Preços do Claude Sonnet 4 (em dólares por milhão de tokens - março/2026)
# ⚠️ Verifique preços atuais em: https://www.anthropic.com/pricing
PRECO_INPUT  = 3.00 / 1_000_000   # $3 por 1M tokens de entrada
PRECO_OUTPUT = 15.00 / 1_000_000  # $15 por 1M tokens de saída

def estimar_custo(texto_entrada: str, tokens_saida_estimados: int = 200) -> tuple:
    """
    Estima o custo de uma chamada à API.
    
    Returns:
        tuple: (tokens_entrada, custo_estimado)
    """
    try:
        contagem = client.messages.count_tokens(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": texto_entrada}]
        )
        tokens_entrada = contagem.input_tokens
        custo = (tokens_entrada * PRECO_INPUT) + (tokens_saida_estimados * PRECO_OUTPUT)
        return tokens_entrada, custo
    except Exception as e:
        print(f"❌ Erro ao estimar custo: {e}")
        return 0, 0.0

cenarios = [
    ("Olá!", "Saudação curta"),
    ("Me explique como funciona a fotossíntese.", "Pergunta simples"),
    ("Resuma em detalhes todos os capítulos de Dom Casmurro de Machado de Assis, " +
     "incluindo análise dos personagens e contexto histórico." * 5, "Prompt longo"),
]

print(f"{'Cenário':<20} {'Tokens entrada':>15} {'Custo estimado':>15}")
print("─" * 52)
for texto, nome in cenarios:
    tokens, custo = estimar_custo(texto)
    print(f"{nome:<20} {tokens:>15} ${custo:.6f}")

print("\n💰 Parecem valores pequenos, mas imagina 1 milhão de usuários!")

---
## 🏆 Desafio — Implemente a função `comparar_idiomas`

Crie uma função que recebe a **mesma frase em português e inglês** e mostra:
- Quantos tokens cada versão usa
- Qual é mais eficiente
- A diferença percentual

**A função está incompleta — complete onde indicado!**

In [ ]:
def comparar_idiomas(texto_pt: str, texto_en: str) -> Dict[str, Any]:
    """
    Compara o número de tokens entre a versão em português e inglês
    de uma mesma frase.
    
    Args:
        texto_pt: frase em português
        texto_en: frase em inglês
    
    Returns:
        dict com tokens_pt, tokens_en, diferenca_percentual, mais_eficiente
    """
    # TODO 1: conte os tokens do texto em português usando client.messages.count_tokens
    # Dica: contagem = client.messages.count_tokens(model=MODEL_NAME, messages=[...])
    # E depois: tokens_pt = contagem.input_tokens
    tokens_pt = None  # ← substitua

    # TODO 2: conte os tokens do texto em inglês (mesmo padrão do TODO 1)
    tokens_en = None  # ← substitua

    # TODO 3: calcule a diferença percentual
    # Dica: diferenca = abs(tokens_pt - tokens_en) / min(tokens_pt, tokens_en) * 100
    diferenca_percentual = None  # ← substitua

    # TODO 4: determine qual idioma é mais eficiente (usa menos tokens)
    mais_eficiente = None  # ← substitua ("português" ou "inglês")

    # TODO 5: imprima um relatório formatado e retorne o dicionário
    # Exemplo de saída esperada:
    # PT: "A inteligência artificial" → 6 tokens
    # EN: "Artificial intelligence"  → 4 tokens
    # Mais eficiente: inglês (33% menos tokens)
    
    return {
        "tokens_pt": tokens_pt,
        "tokens_en": tokens_en,
        "diferenca_percentual": diferenca_percentual,
        "mais_eficiente": mais_eficiente
    }


# ─── Teste sua função com esses exemplos ───
pares = [
    (
        "A inteligência artificial está transformando o mundo",
        "Artificial intelligence is transforming the world"
    ),
    (
        "O gato subiu no telhado e ficou olhando para os pássaros",
        "The cat climbed on the roof and kept watching the birds"
    ),
    (
        "fotossíntese",
        "photosynthesis"
    ),
]

for pt, en in pares:
    resultado = comparar_idiomas(pt, en)
    if resultado:
        print(f"PT: {pt[:40]}... → {resultado.get('tokens_pt')} tokens")
        print(f"EN: {en[:40]}... → {resultado.get('tokens_en')} tokens")
        print(f"Mais eficiente: {resultado.get('mais_eficiente')}")
        print()

---
## 🎯 Desafio Extra (opcional)

Faça uma visualização gráfica comparando o número de tokens de 10 frases em PT vs EN.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pares de frases para comparar
pares_grafico = [
    ("Olá",                  "Hello"),
    ("inteligência",         "intelligence"),
    ("fotossíntese",         "photosynthesis"),
    ("computador",           "computer"),
    ("universidade",         "university"),
    ("matemática",           "mathematics"),
    ("comunicação",          "communication"),
    ("desenvolvimento",      "development"),
    ("responsabilidade",     "responsibility"),
    ("sustentabilidade",     "sustainability"),
]

# TODO: complete o código para gerar o gráfico
# 1. Conte os tokens de cada par
# 2. Crie um gráfico de barras agrupadas (PT vs EN para cada palavra)
# 3. Adicione título, legenda e rótulos nos eixos

rotulos = [pt for pt, _ in pares_grafico]
tokens_pt_lista = []  # ← preencha
tokens_en_lista = []  # ← preencha

# Dica de código para gráfico de barras agrupadas:
x = np.arange(len(rotulos))
largura = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
# barras_pt = ax.bar(x - largura/2, tokens_pt_lista, largura, label='Português', color='steelblue')
# barras_en = ax.bar(x + largura/2, tokens_en_lista, largura, label='Inglês', color='coral')
# ... complete aqui

plt.tight_layout()
plt.show()

---
## 📝 Respostas

Preencha o arquivo `respostas-01.md` com suas observações.

**Perguntas obrigatórias:**
1. O português usa mais ou menos tokens que o inglês em média? Por que isso importa para o custo?
2. Palavras longas e raras usam mais tokens. Por que isso faz sentido considerando como o modelo foi treinado?
3. Se você tem um orçamento de 1.000 tokens de entrada, quantas frases do tipo `"A inteligência artificial..."` cabem?

**Perguntas desafio:**

4. Como você otimizaria um prompt para reduzir o número de tokens sem perder informação?
5. Se um sistema de IA cobra por token, isso cria algum problema de acesso para usuários que falam idiomas com mais tokens?